# 01 - EDA & Feature Engineering

This notebook loads the **real IBM Telco Customer Churn dataset** (downloaded from
Kaggle into `data/raw/`), explores it, and builds the engineered feature table used
by every downstream model.

All paths are built with `os.path.join` so the notebook runs from any working
directory (Jupyter, VS Code, or `jupyter nbconvert`) without path conflicts.

In [1]:
import os
import sys

import pandas as pd


def find_project_root(start: str) -> str:
    """Walk up from *start* until we find the folder containing src/ and config/."""
    current = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(current, "src")) and os.path.isdir(
            os.path.join(current, "config")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError("Could not locate the project root (src/ + config/).")
        current = parent


# Notebooks live in <root>/notebooks, so start the search from the cwd.
PROJECT_ROOT = find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print("PROJECT_ROOT      :", PROJECT_ROOT)
print("RAW_DATA_DIR      :", RAW_DATA_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)

PROJECT_ROOT      : C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System
RAW_DATA_DIR      : C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\data\raw
PROCESSED_DATA_DIR: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\data\processed


## 1. Load the real Kaggle dataset

`load_telco_data()` reads `data/raw/telco_customer_churn.csv` (the real IBM Telco
file populated by `scripts/download_data.py`). If it is missing it downloads it via
`kagglehub` and only falls back to synthetic data as a last resort.

In [2]:
from src.data_pipeline.load_dataset import load_telco_data, telco_raw_path

print("Expected raw file:", telco_raw_path(), "->", os.path.exists(telco_raw_path()))

raw_df = load_telco_data()
print("Raw shape:", raw_df.shape)
raw_df.head()

Expected raw file: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\data\raw\telco_customer_churn.csv -> True
Raw shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
# TotalCharges arrives as text and has blank values for brand-new customers.
print("Blank TotalCharges entries:", (raw_df["TotalCharges"].astype(str).str.strip() == "").sum())
raw_df["TotalCharges"] = pd.to_numeric(raw_df["TotalCharges"], errors="coerce")

print("\nMissing values per column:")
print(raw_df.isnull().sum()[raw_df.isnull().sum() > 0])

Blank TotalCharges entries: 11

Missing values per column:
TotalCharges    11
dtype: int64


## 2. Full exploratory data analysis

`ExploratoryAnalysis.generate_full_eda()` produces summary statistics and saves all
figures to `docs/diagrams/`.

In [5]:
from src.eda.exploratory_analysis import ExploratoryAnalysis

eda = ExploratoryAnalysis(raw_df)
stats = eda.generate_full_eda()

print("Churn rate       :", stats["churn_rate"])
print("Avg tenure (mo)  :", stats["avg_tenure"])
print("Avg monthly chg  :", stats["avg_monthly_charges"])
print("Plots generated  :", stats["plots_generated"])

C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\src\eda\exploratory_analysis.py:303: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\src\eda\exploratory_analysis.py:312: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(


Churn rate       : 0.2654
Avg tenure (mo)  : 32.37
Avg monthly chg  : 64.76
Plots generated  : ['churn_distribution.png', 'feature_distributions.png', 'correlation_matrix.png', 'tenure_distribution.png', 'monthly_charges_churn.png', 'contract_analysis.png', 'churn_by_internetservice.png', 'churn_by_paymentmethod.png', 'churn_by_contract.png']


In [6]:
# Quick business read-out: churn rate by contract, payment method and internet service.
for col in ["Contract", "PaymentMethod", "InternetService"]:
    rate = (
        raw_df.assign(churn=(raw_df["Churn"] == "Yes").astype(int))
        .groupby(col)["churn"]
        .mean()
        .sort_values(ascending=False)
    )
    print(f"\nChurn rate by {col}:")
    print((rate * 100).round(1).astype(str) + " %")


Churn rate by Contract:
Contract
Month-to-month    42.7 %
One year          11.3 %
Two year           2.8 %
Name: churn, dtype: object

Churn rate by PaymentMethod:
PaymentMethod
Electronic check             45.3 %
Mailed check                 19.1 %
Bank transfer (automatic)    16.7 %
Credit card (automatic)      15.2 %
Name: churn, dtype: object

Churn rate by InternetService:
InternetService
Fiber optic    41.9 %
DSL            19.0 %
No              7.4 %
Name: churn, dtype: object


In [7]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

diagram_dir = os.path.join(PROJECT_ROOT, "docs", "diagrams")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, name in zip(
    axes,
    ["churn_distribution.png", "contract_analysis.png", "tenure_distribution.png"],
):
    img_path = os.path.join(diagram_dir, name)
    if os.path.exists(img_path):
        ax.imshow(mpimg.imread(img_path))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

C:\Users\WELCOME\AppData\Local\Temp\ipykernel_26144\3281257183.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Feature engineering

`engineer_features()` adds tenure bands, usage/charge trends, service counts and
risk flags (contract risk, auto-pay, senior-without-support, high-value).

In [8]:
from src.data_pipeline.feature_engineering import engineer_features, prepare_model_data

df = engineer_features(raw_df)

engineered_cols = [c for c in df.columns if c not in raw_df.columns]
print("Engineered columns:")
for c in engineered_cols:
    print("  -", c)
print("\nProcessed shape:", df.shape)
df[engineered_cols].describe().T

Engineered columns:
  - tenure_group
  - avg_monthly_charge
  - charge_increase
  - num_services
  - has_internet
  - has_phone
  - contract_risk
  - auto_pay
  - is_senior_no_support
  - high_value
  - tenure_x_monthly

Processed shape: (7043, 32)


,count,mean,std,min,25%,50%,75%,max
avg_monthly_charge,7032.0,64.799424,30.185891,13.775,36.179891,70.373239,90.179560,121.400
charge_increase,7032.0,-0.001215,2.616165,-18.900,-1.160179,0.000000,1.147775,19.125
num_services,7043.0,4.146244,2.312720,1.000,2.000000,4.000000,6.000000,9.000
has_internet,7043.0,0.783331,0.412004,0.000,1.000000,1.000000,1.000000,1.000
has_phone,7043.0,0.903166,0.295752,0.000,1.000000,1.000000,1.000000,1.000
contract_risk,7043.0,0.550192,0.497510,0.000,0.000000,1.000000,1.000000,1.000
auto_pay,7043.0,0.435326,0.495835,0.000,0.000000,0.000000,1.000000,1.000
is_senior_no_support,7043.0,0.102655,0.303529,0.000,0.000000,0.000000,0.000000,1.000
high_value,7043.0,0.508732,0.499959,0.000,0.000000,1.000000,1.000000,1.000
tenure_x_monthly,7043.0,2279.581350,2264.729447,0.000,394.000000,1393.600000,3786.100000,8550.000


In [9]:
# Sanity-check the model-ready matrix produced from the engineered frame.
X, y = prepare_model_data(df)
print("Model matrix X:", X.shape)
print("Target distribution:", y.value_counts().to_dict())
print("Any residual NaNs:", int(X.isnull().sum().sum()))
X.head()

Model matrix X: (7043, 30)
Target distribution: {0: 5174, 1: 1869}
Any residual NaNs: 22


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,avg_monthly_charge,charge_increase,num_services,has_internet,has_phone,contract_risk,auto_pay,is_senior_no_support,high_value,tenure_x_monthly
0,0,0,1,0,1,0,0,0,0,1,...,29.850000,0.000000,2,1,0,1,0,0,0,29.85
1,1,0,0,0,34,1,0,0,1,0,...,55.573529,1.376471,4,1,1,0,0,0,0,1936.30
2,1,0,0,0,2,1,0,0,1,1,...,54.075000,-0.225000,4,1,1,1,0,0,0,107.70
3,1,0,0,0,45,0,0,0,1,0,...,40.905556,1.394444,4,1,0,0,1,0,0,1903.50
4,0,0,0,0,2,1,0,1,0,0,...,75.825000,-5.125000,2,1,1,1,0,0,1,141.40


## 4. Persist the training snapshot

The processed snapshot is written with `os.path.join` so the modelling notebooks
and the Streamlit app read exactly the same file.

In [10]:
snapshot_path = os.path.join(PROCESSED_DATA_DIR, "telco_churn_processed.csv")
df.to_csv(snapshot_path, index=False)
print("Saved processed snapshot:", snapshot_path)
print("Rows:", len(df), "| Columns:", len(df.columns))

Saved processed snapshot:

 C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\data\processed\telco_churn_processed.csv
Rows: 7043 | Columns: 32
